# PPI-KG
+ GE & PPI-KG overlapping proteins: 8085
+ GE proteins: 8085 + 12480 = 20565
+ PPI-KG proteins: 8085 + 543 = 8628

### BUT the PPI-KG I have can't match the same protein counts  

# Sample Scoring
+ ecdf;
+ healthy control as reference distribution;
+ 1%, 1.5%, 2.5%, 5%, 10%, 20%

# Generate Network
+ my PNG.generate() generate the same results as CLEP's network generator

In [1]:
import os
import sys

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from data_processing.pyg_graph_generator import *

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
output_dir = "../CLEP_repeat/networks/PPI_KGs_my"
dataset = 'adni'
process_method = 'DiseaseKG'
scoring_method = 'ecdf1'

kg_disease_path = "../datasets/base_kgs/ppi_hc.pkl"
kg_health_path = "../data/KG/healthy_aging_reversed_remove_noncausal.pkl"
exp_path = "../data/ADNI/cleaned_gene_expression_data.csv"
scoring_path = "../data/ADNI/old_target/ecdf_1/sample_scoring_ecdf.csv"

os.makedirs(output_dir, exist_ok=True)
save_network = os.path.join(output_dir, f"G_{dataset}_{process_method}_{scoring_method}.pkl")
save_summary = os.path.join(output_dir, f"Summary_{dataset}_{process_method}_{scoring_method}.csv")

In [4]:
# 1. Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(exp_path, index_col=0)
if exp_df.shape[0] > exp_df.shape[1]:
    exp_df = exp_df.transpose()
data = pd.read_csv(scoring_path, index_col=0)
kg_disease = load_graph(kg_disease_path)
kg_control = load_graph(kg_health_path)

# clean exp_df before K-NN
# drop genes with no variation
exp_df = exp_df.loc[:, exp_df.std() > 0]
# Using median is usually safer for gene expression
exp_df = exp_df.fillna(exp_df.median())
# normalize safely
min_val = exp_df.min()
max_val = exp_df.max()
exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
# final fill-na
exp_norm = exp_norm.fillna(0)

Loaded graph from ../datasets/base_kgs/ppi_hc.pkl: 17042 nodes, 382526 edges
Loaded graph from ../data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13775 edges


# RotatE 
+ best hyperparameters are in the config dict

In [48]:
2**6

64

In [ ]:
# best hyperparameters for RotatE

best_config = {
    "model": "RotatE",
    "training_loop": "sLCWA",  # Stochastic Local Closed World Assumption
    
    # Model-specific parameters
    "model_kwargs": {
        "embedding_dim": 128,  
    },
    
    # Loss function and its specific parameters
    "loss": "NSSALoss",  # Negative Sampling Self-Adversarial Loss
    "loss_kwargs": {
        "margin": 23.92,
        "adversarial_temperature": 0.93,
    },
    
    # Negative Sampler configuration (used by sLCWA)
    "negative_sampler_kwargs": {
        "num_negatives_per_positive": 22,
    },
    
    # Training runtime execution parameters
    "training_kwargs": {
        "num_epochs": 200,
        "batch_size": 1024,
    }
}

In [7]:
from CLEP_repeat.embedding.kge import *

In [23]:
graph = load_graph("../CLEP_repeat/networks/PPI_KGs_my/G_adni_DiseaseKG_ecdf_1.pkl")
design_path = "../data/ADNI/design_with_real_target.tsv"
design = pd.read_csv(design_path, index_col=0, sep='\t')
design['Target'] = design['Old_Target'].map({'Disease':1, 'Control':0})
design

Loaded graph from ../CLEP_repeat/networks/PPI_KGs_my/G_adni_DiseaseKG_ecdf_1.pkl: 17786 nodes, 611876 edges


,Old_Target,Target
FileName,,
116_S_1249,Control,0
037_S_4410,Control,0
006_S_4153,Disease,1
116_S_1232,Control,0
099_S_4205,Disease,1
...,...,...
009_S_2381,Disease,1
053_S_4557,Disease,1
073_S_4300,Disease,1


In [36]:
design['Sample_ID'] = design.index
design['Sample_ID2'] = design['Sample_ID'].str.extract(r'(\d{3}_S_\d{4})')
design['Sample_ID2'].isna().sum()

0

In [28]:
graph_df = nx.to_pandas_edgelist(graph)
graph_df

,source,target,weight,confidence,rel,evidence,relation
0,AL1A1_HUMAN,AL1A1_HUMAN,NaN,0.76,PPI,"experiments:in vivo,Two-hybrid;pmids:12081471,...",NaN
1,AL1A1_HUMAN,ALDH2_HUMAN,NaN,0.82,PPI,"experiments:Two-hybrid,cross-linking study;pmi...",NaN
2,AL1A1_HUMAN,AL1A2_HUMAN,NaN,0.72,PPI,experiments:affinity chromatography technology...,NaN
3,AL1A1_HUMAN,P4K2A_HUMAN,NaN,0.72,PPI,"experiments:anti tag coimmunoprecipitation,aff...",NaN
4,ITA7_HUMAN,ACHA_HUMAN,NaN,0.73,PPI,"experiments:in vivo,Affinity Capture-Western,a...",NaN
...,...,...,...,...,...,...,...
611871,007_S_0101,YPEL2_HUMAN,0.127827,NaN,NaN,NaN,down_reg
611872,007_S_0101,ZBP1_HUMAN,0.862738,NaN,NaN,NaN,up_reg
611873,007_S_0101,ZFP41_HUMAN,0.739417,NaN,NaN,NaN,up_reg
611874,007_S_0101,ZNFX1_HUMAN,0.895191,NaN,NaN,NaN,up_reg


In [40]:
graph_df['relation'] = graph_df['relation'].fillna('PPI')
graph_df=graph_df[['source','target','relation']]
graph_df['Sample_ID'] = graph_df['source'].str.extract(r'(\d{3}_S_\d{4})')
graph_df['label'] = graph_df['Sample_ID'].map(design['Target'])
graph_df = graph_df.drop(columns=['Sample_ID'])
graph_df

,source,target,relation,label
0,AL1A1_HUMAN,AL1A1_HUMAN,PPI,NaN
1,AL1A1_HUMAN,ALDH2_HUMAN,PPI,NaN
2,AL1A1_HUMAN,AL1A2_HUMAN,PPI,NaN
3,AL1A1_HUMAN,P4K2A_HUMAN,PPI,NaN
4,ITA7_HUMAN,ACHA_HUMAN,PPI,NaN
...,...,...,...,...
611871,007_S_0101,YPEL2_HUMAN,down_reg,1.0
611872,007_S_0101,ZBP1_HUMAN,up_reg,1.0
611873,007_S_0101,ZFP41_HUMAN,up_reg,1.0
611874,007_S_0101,ZNFX1_HUMAN,up_reg,1.0


### KGE

In [43]:
def weighted_splitter(
        edgelist: pd.DataFrame,
        train_size: float = 0.8,
        validation_size: float = 0.1
) -> Tuple[pd.DataFrame, ...]:
    """Split the given edgelist into training, validation and testing sets on the basis of the ratio of relations.

    :param edgelist: Edgelist in the form of (Source, Relation, Target)
    :param train_size: Size of the training data
    :param validation_size: Size of the training data
    :return: Tuple containing the train, validation & test splits
    """
    # Validation size is the size of the percentage of the remaining data (i.e. If required validation size is 10% of
    # the original data & training size is 80% then the new validation size is 50% of the data without the training
    # data. The similar calculation is done for training size, hence it is always 1
    validation_size = validation_size / (1 - train_size)
    test_size = 1

    # Get the unique relations in the network
    unique_relations = sorted(edgelist['relation'].unique())

    data = edgelist.drop_duplicates().copy()

    split = []
    # Split the data to get training, validation and test samples
    for frac_size in [train_size, validation_size, test_size]:
        frames = []
        # Random sampling of the data for every type of relation
        for relation in unique_relations:
            temp = data[data['relation'] == relation].sample(frac=frac_size)

            data = data[~data.index.isin(temp.index)]

            frames.append(temp)
        # Join all the different relations in one dataframe
        split.append(pd.concat(frames, ignore_index=True, sort=False))

    return tuple(split)

In [62]:
hpo_config = {
    # 1. Wrap all model and training specs into the "pipeline" key
    "pipeline": {
        "model": "RotatE",
        "training_loop": "sLCWA",
        
        "model_kwargs": {
            "embedding_dim": 128,  
        },
        
        "loss": "NSSALoss",
        "loss_kwargs_ranges": {
            "margin": {
                "type": "float",
                "low": 15.0,
                "high": 30.0,
                "scale": "linear"
            },
            "adversarial_temperature": {
                "type": "float",
                "low": 0.5,
                "high": 1.5,
                "scale": "linear"
            }
        },

        "negative_sampler": "basic",
        "negative_sampler_kwargs_ranges": {
            "num_negs_per_pos": {
                "type": "int",
                "low": 10,
                "high": 30,
                "step": 2
            }
        },

        "optimizer": "Adam",
        "optimizer_kwargs_ranges": {
            "lr": {
                "type": "float",
                "low": 0.0001,
                "high": 0.01,
                "scale": "log"
            }
        },

        "training_kwargs": {
            "num_epochs": 20,
        },
        "training_kwargs_ranges": {
            "batch_size": {
                "type": "categorical",
                "choices": [512, 1024, 2048]
            }
        },

        "evaluator": "RankBasedEvaluator",
        "evaluator_kwargs": {
            "filtered": True
        },
        "evaluation_kwargs": {
            "batch_size": 1024
        }
    },
    
    # 2. mandatory "optuna" key to control the HPO mechanics
    "optuna": {
        "n_trials": 2,       # Number of parameter combinations to try
        "timeout": 3600,      # Optional: Max time in seconds (1 hour)
        "metric": "hits_at_10", # What metric to maximize (defaults to mean_reciprocal_rank if left out)
        "direction": "maximize" 
    }
}

In [54]:
edgelist = graph_df
out = './kge_result'
train_size = 0.8
validation_size = 0.1

os.makedirs(out, exist_ok=True)

In [55]:
design_norm_df = design.astype(str, copy=True)

unique_nodes = edgelist[~edgelist['label'].isna()].drop_duplicates('source')

# Create a mapping of the patient to the label. The patient id is converted to string to avoid duplicates
label_mapping = {str(patient): label for patient, label in zip(unique_nodes['source'], unique_nodes['label'])}

edgelist = edgelist.drop(columns='label')

# Split the edgelist into training, validation and testing data
train, validation, test = weighted_splitter(
    edgelist=edgelist,
    train_size=train_size,
    validation_size=validation_size
)

train_path = os.path.join(out, 'train.edgelist')
validation_path = os.path.join(out, 'validation.edgelist')
test_path = os.path.join(out, 'test.edgelist')

train.to_csv(train_path, sep='\t', index=False, header=False)
validation.to_csv(validation_path, sep='\t', index=False, header=False)
test.to_csv(test_path, sep='\t', index=False, header=False)

/var/folders/ft/2nftg5q91n5dhpf5dtts4gkm0000gn/T/ipykernel_10328/2221185373.py:1: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  design_norm_df = design.astype(str, copy=True)


In [ ]:
from pykeen.hpo.hpo import hpo_pipeline_from_config

train_tf = TriplesFactory.from_path(train_path, create_inverse_triples=True)
validation_tf = TriplesFactory.from_path(validation_path, create_inverse_triples=True)
test_tf = TriplesFactory.from_path(test_path, create_inverse_triples=True)

# Define HPO pipeline
hpo_results = hpo_pipeline_from_config(
    dataset=None,
    training=train_tf,
    testing=test_tf,
    validation=validation_tf,
    config=hpo_config
)

optimization_dir = os.path.join(out, 'pykeen_results_optim')
if not os.path.isdir(optimization_dir):
    os.makedirs(optimization_dir)

hpo_results.save_to_directory(optimization_dir)

[I 2026-06-01 11:15:51,326] A new study created in memory with name: no-name-aaba25b3-337b-4160-9dfa-dd5feadee4ee
INFO:pykeen.hpo.hpo:Using model: <class 'pykeen.models.unimodal.rotate.RotatE'>
INFO:pykeen.hpo.hpo:Using loss: <class 'pykeen.losses.NSSALoss'>
INFO:pykeen.hpo.hpo:Using optimizer: <class 'torch.optim.adam.Adam'>
INFO:pykeen.hpo.hpo:Using training loop: <class 'pykeen.training.slcwa.SLCWATrainingLoop'>
INFO:pykeen.hpo.hpo:Using negative sampler: <class 'pykeen.sampling.basic_negative_sampler.BasicNegativeSampler'>
INFO:pykeen.hpo.hpo:Using evaluator: <class 'pykeen.evaluation.rank_based_evaluator.RankBasedEvaluator'>
INFO:pykeen.hpo.hpo:Attempting to maximize both.realistic.hits_at_10
INFO:pykeen.hpo.hpo:Filter validation triples when testing: True
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.triples.triples_factory:Creating inv

In [ ]:
pipeline_results = run_pipeline(
    out_dir=out
)

best_model, triple_factory = pipeline_results.model, pipeline_results.training

# Get the embedding as a numpy array. Ignore the type as the model will be of type ERModel (Embedding model)
embedding_values = _model_to_numpy(best_model, complex=complex_embedding)  # type: ignore

# Create columns as component names
embedding_columns = [f'Component_{i}' for i in range(1, embedding_values.shape[1] + 1)]

# Get the nodes of the training triples as index
node_list = list(triple_factory.entity_to_id.keys())
embedding_index = sorted(node_list, key=lambda x: triple_factory.entity_to_id[x])

embedding = pd.DataFrame(data=embedding_values, columns=embedding_columns, index=embedding_index)

if return_patients:
    # TODO: Use clustering before classification to see if embeddings are already good enough
    embedding = embedding[embedding.index.isin(design_norm_df['FileName'])]

    for index in embedding.index:
        embedding.at[index, 'label'] = label_mapping[index]


# Classification: Logistic Regression